# Notebook to fit all available scans with MEpHISDO

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

from hypermri import BrukerDir
from hypermri.sequences import umpireMGE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style
import matplotlib as mpl
from hypermri.utils.utils_interpolation import interpolate_3d_array
from scipy.optimize import Bounds

## Create all uMGE objects

In [5]:
def preprocess_uMGE_object(scan_list, hyper_nr, snr_thr):
    
    uMGE_object = umpireMGE(scan_list[hyper_nr],self_reco=True, channel_combination_method = "sum",remove_zeropadding=True)
    
    uMGE_object.mask_snr_silent(thr=snr_thr, factor=0.05, echoNr=0, use_k_space=True)
    
    threshold_frequency_proximity = 25
    f_search_range = 10
    t2_star_boundary =  Bounds([0.01],[0.1])
    snr_threshold = 7.5
    frequ_inits=np.array([250.0, 190.0, 100.0])
    t2_star_init = 0.015
    
    
    uMGE_object.MEpHISDO(frequ_inits=frequ_inits, threshold_frequency_proximity = threshold_frequency_proximity,
                         t2_star_init=t2_star_init, f_search_range=f_search_range, t2_star_boundary=t2_star_boundary, snr_threshold=snr_threshold)
    
    return uMGE_object

def create_MEpHISDO_maps(uMGE_object, field_map_object, data_path, uMGE_object_name, c1_reference, snr_thr):

    
    print(field_map_object.fmap[9,5,5])
    _ = uMGE_object.apply_1H_ref_map_multi_comp(field_map_object.fmap, mask=field_map_object.fmap.mask)
                
    uMGE_object.calculate_pH(c1_center=c1_reference)
    uMGE_object.calculate_peak_pH_map()
    
    OVERLAY_SHAPE = list(uMGE_object.data[0].shape)
    OVERLAY_SHAPE[0] = 20
    
    uMGE_object._fit_cost_map = uMGE_object._fit_cost_map / np.max(np.abs(uMGE_object.data), axis=0)

    uMGE_object.interpolate_afterwards(OVERLAY_SHAPE, method='nearest')
    uMGE_object.mask_snr_silent(thr=snr_thr, factor=0.05, echoNr=0, use_k_space=True)
    

    uMGE_object.mask_cost_error(cost_error_thr=1.6)
    
    uMGE_object.save(data_path+uMGE_object_name)
    


In [ ]:
df = pd.read_excel("./OVERVIEW_SequenceComparison_September2025_noresults.xlsx")
df.set_index('Scan_Nr', inplace=True)

In [ ]:
snr_thresholds= np.array([10,10,10,10,10,10,10,10])

field_map_name = "umge_field_map_lr_shifted_wo_zeros"

save_path = r"./data/"

for scan in range(0,1):
    
    META = df.loc[scan+1]
    raw_data_path = META['Raw_Data_Path']
    
    # load field map 
    field_map_object = umpireMGE.load("./" + field_map_name + ".pickle")

    uMGE_name = "MEpHISDO_object_scan_mean_combi_" + str(scan+1) + ".pickle"

    scans = BrukerDir(raw_data_path,verbose =False)

    preprocessed_uMGE_object = preprocess_uMGE_object(scans, META['ExpNr_uMGE_Hyper'], snr_thresholds[scan])

    create_MEpHISDO_maps(preprocessed_uMGE_object, field_map_object, save_path, uMGE_name, META['C1_Frequency'], snr_thresholds[scan])

  0%|          | 0/30 [00:00<?, ?it/s]

[ERROR] hypermri.brukerexp.Load_fid_file(): Both 'fid' and 'ser' files do not exist or are empty for scan 9. 
C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\sequences\base_anatomy.py:82: UserWarning: Could not reorient seq2d.
  warning.warn("Could not reorient seq2d.")
[ERROR] hypermri.brukerdir: The following error occured when trying to load in scan 9 as 'umpireMGE':
    IndexError: list index out of range
 
[CRITICAL] hypermri.sequences.CSI: [ExpNum=10] The following error occured during shift_kspace():
    Exception: Shift not yet implemented for  read orientation: H_Ffor slice_orient: coronal
 
C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\sequences\base_anatomy.py:82: UserWarning: Could not reorient seq2d.
  warning.warn("Could not reorient seq2d.")
C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\sequences\base_anatomy.py:82: UserWarning: Could not reorient seq2d.
  warning.wa

Data scaled by: 23983989.23263084
Mean noise:                      -0.00 + 0.00*1i
Mean |noise|:                    0.01714873882299818
Standard deviation of noise: σ = 0.01 + 0.01*1i
Rayleigh Fit of |noise|:     σ = 0.01
Std.Dev.: 0.88
SNR Min.: 0.05
SNR Max.: 127.44


C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\utils\utils_general.py:3078: UserWarning: No mask provided, using all voxels for noise estimation!
  warnings.warn("No mask provided, using all voxels for noise estimation!")


Calculating Voxels:   0%|          | 0/3564 [00:00<?, ?it/s]

C:\Users\Admin\anaconda3\envs\hypermri_env_oneDrive\lib\site-packages\joblib\externals\loky\process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[WARNING] hypermri.sequences.umpireMGE.umpireMGE: Replacing masked values with zero... 
C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\sequences\umpireMGE\umpireMGE.py:728: RuntimeWarning: invalid value encountered in log10
  return pK + np.log10(log_arg)
[ERROR] hypermri.sequences.umpireMGE.umpireMGE: The pH conversion failed, all entries are NaN. 


... (3 outputs left out)

Data scaled by: 20767069.36257715
Mean noise:                      0.00 + -0.00*1i
Mean |noise|:                    0.020138626222934
Standard deviation of noise: σ = 0.02 + 0.02*1i
Rayleigh Fit of |noise|:     σ = 0.02
Std.Dev.: 0.98
SNR Min.: 0.12
SNR Max.: 123.73


C:\Users\Admin\OneDrive - TUM\Dokumente\Coding\python-code-bruker-7t\src\hypermri\utils\utils_general.py:3078: UserWarning: No mask provided, using all voxels for noise estimation!
  warnings.warn("No mask provided, using all voxels for noise estimation!")


Calculating Voxels:   0%|          | 0/3240 [00:00<?, ?it/s]

KeyboardInterrupt: 